# Build Fact Payments
1. read the data from the silver order_payments table
2. join silver orders table, dim_customers and dim_date tables
3. select the required columns
- Measures: payment_value
- Foreign keys: customer_sk, payment_date_key
- Degenerate Dimensions: order_id, payment_sequential, payment_type, payment_installments
4. write the transformed data to gold fact_order_payments table

In [0]:
#Imports
from pyspark.sql.functions import col,cast,date_diff

### Step1 - read the data from the silver order_payments table

In [0]:
order_payments_df = spark.read.table("olist_catalog.silver.order_payments")
display(order_payments_df)

### Step2 - join silver orders table, join silver orders table, dim_customers and dim_date tables

In [0]:
orders_df = spark.read.table("olist_catalog.silver.orders")
orders_df = (
    orders_df.select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_purchase_timestamp").cast("date").alias("order_purchase_date"),
        col("order_approved_at").cast("date").alias("order_approved_date"),
        col("order_delivered_carrier_date").cast("date"),
        col("order_delivered_customer_date").cast("date"),
        col("order_estimated_delivery_date").cast("date"),
        date_diff(col("order_delivered_customer_date"),col("order_purchase_date")).alias("delivery_days"),
        date_diff(col("order_estimated_delivery_date"),col("order_purchase_date")).alias("estimated_delivery_days")
    )
)

In [0]:
dim_date_df = spark.read.table("olist_catalog.gold.dim_date")
dim_customers_df = spark.read.table("olist_catalog.gold.dim_customers")

In [0]:
fact_order_payments_df = (
    order_payments_df.alias("op")
    .join(orders_df.alias("o"),
          col("o.order_id")==col("op.order_id"),
          "left"
          )
    .join(dim_date_df.alias("d"),
          col("o.order_purchase_date")==col("d.full_date"),
          "left"
          )
    .join(dim_customers_df.alias("c"),
          col("o.customer_id")==col("c.customer_id")
          
          )
    )

### Step3 - select the required columns 


In [0]:
fact_order_payments_df=(
    fact_order_payments_df.select(
        "op.order_id",
        "op.payment_sequential",
        "op.payment_type",
        "op.payment_installments",
        "c.customer_sk",
        col("d.date_key").alias("payment_date_key"),
        "op.payment_value"
    )
    )

### Step4 - write the transformed data to gold fact_order_payments table

In [0]:
(
    fact_order_payments_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.gold.fact_order_payments")
)